## Step 1: Import Required Libraries

First, import all the classes needed to create an Auto-Merging Retriever pipeline.

- **StorageContext** → Stores documents and nodes.
- **ServiceContext** → Holds LLM, embedding model, and parser settings (optional here).
- **VectorStoreIndex** → Creates a vector index from leaf nodes.
- **HierarchicalNodeParser** → Splits documents into parent and child chunks.
- **get_leaf_nodes()** → Extracts the smallest chunks for indexing.
- **AutoMergingRetriever** → Automatically merges retrieved child nodes into parent nodes.
- **RetrieverQueryEngine** → Uses the retriever to answer questions.
- **SentenceTransformerRerank** → Optional reranker to improve retrieved results.

In [1]:
from llama_index.core import StorageContext, ServiceContext, VectorStoreIndex
from llama_index.core.node_parser import (
    HierarchicalNodeParser,
    get_leaf_nodes
)
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.postprocessor import SentenceTransformerRerank

## Step 2: Create a Hierarchical Node Parser

Instead of splitting the document into one chunk size, we create multiple levels.

Example:

Level 1 → 1024 tokens (Parent)

Level 2 → 512 tokens (Child)

Level 3 → 128 tokens (Leaf)

The retriever will search only the smallest (leaf) chunks, but it can automatically merge them back into larger parent chunks when needed.

In [2]:
# Create a hierarchical text splitter
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[1024, 512, 128]   # Parent → Child → Leaf
)

### Load Documents

Load one or more documents using **SimpleDirectoryReader**. The loaded documents will later be split into nodes and indexed for retrieval.

In [7]:
from llama_index.readers.web import SimpleWebPageReader

# Load content directly from a webpage
documents = SimpleWebPageReader().load_data(
    urls=["https://lilianweng.github.io/posts/2023-06-23-agent/"]
)

In [8]:
len(documents)

1

## Step 3: Convert Documents into Hierarchical Nodes

The parser converts each document into a tree structure.

Document
│
├── Parent Node (1024)
│
├── Child Node (512)
│
├── Leaf Node (128)
│
└── Leaf Node (128)

Only the leaf nodes are indexed for similarity search.

In [10]:
# Split documents into hierarchical nodes
nodes = node_parser.get_nodes_from_documents(documents)

# Extract only the smallest chunks (leaf nodes)
leaf_nodes = get_leaf_nodes(nodes)

In [12]:
len(leaf_nodes)

455

In [13]:
print(leaf_nodes[0])

Node ID: e5973a99-998c-4097-b989-76751de0ac58
Text: <!DOCTYPE html> <html lang="en-us" dir="auto">  <head><meta
charset="utf-8"> <meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta name="viewport" content="width=device-width, initial-scale=1,
shrink-to-fit=no"> <meta name="robots" content="index, follow">
<title>LLM Powered Autonomous Agents | Lil&#39;Log</title> <meta
name="keywords" con...


### Initialize the Embedding Model

Initialize the **HuggingFace BGE embedding model**, which converts text into vector embeddings for semantic search.

- **Model:** `BAAI/bge-small-en-v1.5`
- **normalize_embeddings=True:** Normalizes vectors so cosine similarity can be used effectively during retrieval.

In [14]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Name of the Hugging Face embedding model
model_name = "BAAI/bge-small-en-v1.5"

# Normalize embeddings for cosine similarity
encode_kwargs = {
    "normalize_embeddings": True
}

# Initialize the embedding model
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    # model_kwargs={"device": "cuda"},  # Uncomment to run on GPU
    encode_kwargs=encode_kwargs,
)

In [17]:
from llama_index.embeddings.langchain import LangchainEmbedding

## Step 4: Create Storage Context and Vector Index

StorageContext stores all nodes (parents + children).

The VectorStoreIndex is built using only the leaf nodes because searching smaller chunks is more accurate.

Why store parent nodes?

Auto-Merging Retriever needs them later to reconstruct larger context.

In [18]:
# Create storage for all hierarchical nodes
storage_context = StorageContext.from_defaults()

# Store all nodes (parent and child nodes)
storage_context.docstore.add_documents(nodes)

# Build the vector index using the leaf nodes
index = VectorStoreIndex(
    leaf_nodes,
    storage_context=storage_context,
    embed_model=LangchainEmbedding(embedding_function)
)

In [23]:
print(type(index))

<class 'llama_index.core.indices.vector_store.base.VectorStoreIndex'>


In [24]:
print(dir(index))

['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__implemented__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__providedBy__', '__provides__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_nodes_to_index', '_adelete_from_docstore', '_adelete_from_index_struct', '_aget_node_with_embedding', '_async_add_nodes_to_index', '_build_index_from_nodes', '_callback_manager', '_delete_from_docstore', '_delete_from_index_struct', '_delete_node', '_docstore', '_embed_model', '_get_node_with_embedding', '_graph_store', '_index_struct', '_insert', '_insert_batch_size', '_is_protocol', '_object_map', '_show_progress', '_storage_context', '_store_nodes_

In [26]:
print(len(index.storage_context.docstore.docs))

592


In [27]:
first_node = next(iter(index.storage_context.docstore.docs.values()))

print(first_node.text)

<!DOCTYPE html>
<html lang="en-us" dir="auto">

<head><meta charset="utf-8">
<meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
<meta name="robots" content="index, follow">
<title>LLM Powered Autonomous Agents | Lil&#39;Log</title>
<meta name="keywords" content="nlp, language-model, agent, steerability, prompting" />
<meta name="description" content="Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview
In a LLM-powered autonomous agent system, LLM functions as the agent&rsquo;s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down 

## Step 5: Create the Auto-Merging Retriever

First, perform normal similarity search on the leaf nodes.

Then, AutoMergingRetriever checks whether enough child nodes from the same parent were retrieved.

If yes, it replaces them with the larger parent node.

Example:

Retrieved:

✓ Child 1

✓ Child 2

✓ Child 3

↓

Merged into:

✓ Parent Node

In [28]:
# Create the basic similarity retriever
base_retriever = index.as_retriever(
    similarity_top_k=6   # Retrieve top 6 leaf nodes
)

# Automatically merge nearby child nodes into their parent node
retriever = AutoMergingRetriever(
    base_retriever,
    storage_context,
    verbose=True          # Print merging information
)

## Step 6: Create the Query Engine

The RetrieverQueryEngine connects the retriever with the LLM.

Workflow:

User Question

↓

Retriever

↓

Relevant Nodes

↓

LLM

↓

Final Answer

In [33]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [34]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core import Settings

# Configure the Gemini LLM
Settings.llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
)

In [37]:
import nest_asyncio

# Allow nested asyncio event loops in Jupyter
nest_asyncio.apply()

In [38]:
# Build the query engine using the Auto-Merging Retriever
query_engine = RetrieverQueryEngine.from_args(
    retriever
)

## Step 7: Ask a Question

The query engine performs the following steps:

1. Convert the question into an embedding.
2. Search similar leaf nodes.
3. Merge child nodes into parents when appropriate.
4. Send the retrieved context to the LLM.
5. Generate the final answer.

In [39]:
# Ask a question
response = query_engine.query(
    "What are the different Chain of Thought prompting?"
)

# Display the generated answer
print(response)

Chain of Thought (CoT) prompting is a technique that encourages large language models to think step by step, breaking down complex tasks into simpler, manageable stages.

Extensions of this approach include:
*   **Tree of Thoughts (ToT)**: This method expands on CoT by exploring multiple reasoning possibilities at each step of the problem-solving process.
*   **ReAct**: This prompting template integrates explicit steps for the model to follow, formatted as "Thought: ... Action: ... Observation: ...", which can be repeated. This allows the model to interact with environments and generate natural language reasoning traces.


### Workflow

```text
Documents
     │
     ▼
HierarchicalNodeParser
     │
     ▼
Parent Nodes (1024)
     │
     ▼
Child Nodes (512)
     │
     ▼
Leaf Nodes (128)
     │
     ▼
VectorStoreIndex
     │
     ▼
Similarity Search
     │
     ▼
AutoMergingRetriever
(Merges nearby leaf nodes into parent nodes)
     │
     ▼
RetrieverQueryEngine
     │
     ▼
LLM
     │
     ▼
Final Answer
```